
# Rule‑First Fraud Detection (Synthetic FD) — EDA → Feature Engineering → XGBoost

This notebook implements a **rule-first** approach tailored for the provided synthetic dataset and any data that shares the same pattern.

**Key insight (exact patterns found):**
- **Fraudulent TRANSFERs** satisfy:  
  `oldbalanceDest == 0` **and** `(oldbalanceOrg - newbalanceOrig - amount) == 0`  
  (i.e., the sender's account is debited exactly by `amount`, and the destination had a zero starting balance.)
- **Fraudulent CASH_OUTs (dominant pattern)** satisfy:  
  `newbalanceOrig == 0` **and** `amount == oldbalanceOrg`  
  (i.e., cash-out fully empties the origin account.)

We encode these as **rule features** (no use of `isFraud` or `isFlaggedFraud` in the logic), then train an **XGBoost** model on the larger dataset.  
**Prediction time behavior:** we first apply the rules; if a transaction matches, it's flagged as fraud. Otherwise, XGBoost decides.


In [ ]:

# --- Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional ML imports (installed in many environments)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib

# Paths
SAMPLE_PATH = Path('/mnt/data/Synthetic_FD.csv')  # sample dataset (small)
LARGE_PATH = Path('/mnt/data/FD_large.csv')       # TODO: replace with your 6.36M rows file path

# Load sample (for EDA sanity checks)
df = pd.read_csv(SAMPLE_PATH)
print(df.shape)
df.head()


In [ ]:

# --- Basic EDA
print("Columns:", df.columns.tolist())
print("\nTypes:\n", df.dtypes)
print("\nNulls:\n", df.isna().sum())
print("\nType distribution:\n", df['type'].value_counts(dropna=False))

# Quick numeric summary
df.describe(include='all')


In [ ]:

# --- Helper: add interpretable bookkeeping features (NO target used here)
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    # Balance/bookkeeping deltas
    d['orig_delta'] = d['oldbalanceOrg'] - d['newbalanceOrig'] - d['amount']
    d['dest_delta'] = d['newbalanceDest'] - d['oldbalanceDest'] - d['amount']
    # Common signals
    d['amountEqualsOrigBalance'] = (np.isclose(d['amount'], d['oldbalanceOrg'])).astype(int)
    d['newOrigZero'] = (np.isclose(d['newbalanceOrig'], 0)).astype(int)
    d['oldDestZero'] = (np.isclose(d['oldbalanceDest'], 0)).astype(int)
    d['newDestZero'] = (np.isclose(d['newbalanceDest'], 0)).astype(int)
    # Log amount
    d['logAmount'] = np.log1p(d['amount'])
    # Type flags
    d['isTransfer'] = (d['type'] == 'TRANSFER').astype(int)
    d['isCashOut'] = (d['type'] == 'CASH_OUT').astype(int)
    # --- RULES (do not use isFraud/isFlaggedFraud) ---
    # Fraud-TRANSFER rule
    d['rule_fraud_transfer'] = ((d['isTransfer'] == 1) &
                                (np.isclose(d['oldbalanceDest'], 0)) &
                                (np.isclose(d['orig_delta'], 0))).astype(int)
    # Fraud-CASH_OUT rule (dominant pattern)
    d['rule_fraud_cashout'] = ((d['isCashOut'] == 1) &
                               (np.isclose(d['newbalanceOrig'], 0)) &
                               (np.isclose(d['amount'], d['oldbalanceOrg']))).astype(int)
    # Combined rule
    d['rule_fraud'] = ((d['rule_fraud_transfer'] == 1) | (d['rule_fraud_cashout'] == 1)).astype(int)
    return d

df_feat = add_features(df)
df_feat.head()


In [ ]:

# --- Visual sanity checks (matplotlib only; one plot per figure, no custom colors)
plt.figure()
df_feat['type'].value_counts().plot(kind='bar', title='Type counts')
plt.show()

plt.figure()
df_feat['rule_fraud'].value_counts().plot(kind='bar', title='Rule Fraud Flag counts (sample)')
plt.show()


In [ ]:

# --- On the small sample: how well do rules alone do? (for awareness only)
if 'isFraud' in df_feat.columns:
    y_true = df_feat['isFraud'].astype(int)
    y_rule = df_feat['rule_fraud'].astype(int)
    print("Confusion matrix (Rules only on sample):")
    print(confusion_matrix(y_true, y_rule))
    print("\nReport:")
    print(classification_report(y_true, y_rule, digits=4))



## Train XGBoost on Large Dataset (Rule‑First)

We train a model **only** to decide among the **ambiguous** transactions (those not caught by rules).  
At prediction time, we first apply the rules; if they fire, we directly flag as fraud; otherwise, we ask the model.


In [ ]:

# --- Load large dataset
if LARGE_PATH.exists():
    big = pd.read_csv(LARGE_PATH)
else:
    # Fallback to sample if large file not present (keeps notebook runnable)
    print("WARNING: LARGE_PATH not found — using SAMPLE_PATH for demonstration.")
    big = df.copy()

big_feat = add_features(big)

# Train only on ambiguous rows (rule_fraud == 0), but evaluate end-to-end by OR-ing rule with model.
X_all = big_feat.drop(columns=['isFraud','isFlaggedFraud'], errors='ignore')

# Columns to model (exclude labels and rule outputs that trivially leak label)
exclude_cols = {'isFraud','isFlaggedFraud','rule_fraud','rule_fraud_transfer','rule_fraud_cashout'}
feature_cols = [c for c in big_feat.columns if c not in exclude_cols]

# Train/val split
if 'isFraud' in big_feat.columns:
    y_all = big_feat['isFraud'].astype(int).values
else:
    # If the large dataset lacks labels, the pipeline still runs; only prediction is possible.
    y_all = None

# Mask ambiguous rows
amb_mask = (big_feat['rule_fraud'] == 0).values
X_amb = big_feat.loc[amb_mask, feature_cols].copy()
y_amb = big_feat.loc[amb_mask, 'isFraud'].astype(int).values if y_all is not None else None

# Preprocess: One-hot encode 'type' only; others numeric
cat_cols = ['type']
num_cols = [c for c in feature_cols if c not in cat_cols]

pre = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_cols),
    ('num', 'passthrough', num_cols)
])

# Model
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    eval_metric='logloss',
)

pipe = Pipeline(steps=[('pre', pre), ('xgb', xgb)])

if y_amb is not None and len(X_amb) > 0:
    X_train, X_val, y_train, y_val = train_test_split(X_amb, y_amb, test_size=0.2, random_state=42, stratify=y_amb)
    pipe.fit(X_train, y_train)
    print("Validation on ambiguous rows only:")
    y_val_pred = pipe.predict(X_val)
    print(confusion_matrix(y_val, y_val_pred))
    print(classification_report(y_val, y_val_pred, digits=4))
else:
    print("No labels or no ambiguous rows — skipping training.")


In [ ]:

# --- End-to-end evaluation: rule-first + model on big dataset (requires labels to report metrics)
def predict_rule_plus_model(df_raw: pd.DataFrame, model: Pipeline):
    d = add_features(df_raw)
    # Rule prediction
    y_rule = d['rule_fraud'].astype(int).values
    # Model only for ambiguous
    amb_mask = (y_rule == 0)
    y_model = np.zeros_like(y_rule)
    if model is not None and hasattr(model, 'predict') and amb_mask.any():
        X_amb = d.loc[amb_mask, feature_cols]
        y_model_part = model.predict(X_amb)
        y_model[amb_mask] = y_model_part
    # Final
    y_pred = np.where(y_rule==1, 1, y_model)
    return y_pred, d

if 'isFraud' in big_feat.columns:
    y_pred, d_all = predict_rule_plus_model(big, pipe if 'pipe' in globals() else None)
    print("End-to-end (Rules OR Model) on big dataset:")
    print(confusion_matrix(big_feat['isFraud'].astype(int).values, y_pred))
    print(classification_report(big_feat['isFraud'].astype(int).values, y_pred, digits=4))
else:
    print("No labels on large dataset — skipping end-to-end evaluation.")


In [ ]:

# --- Save pipeline and a lightweight inference function
OUT_DIR = Path('/mnt/data/fd_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, OUT_DIR / 'xgb_rule_first.pkl')

# Save a small inference-only module
infer_code = f"""
import numpy as np
import pandas as pd
import joblib

def add_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    d['orig_delta'] = d['oldbalanceOrg'] - d['newbalanceOrig'] - d['amount']
    d['dest_delta'] = d['newbalanceDest'] - d['oldbalanceDest'] - d['amount']
    d['amountEqualsOrigBalance'] = (np.isclose(d['amount'], d['oldbalanceOrg'])).astype(int)
    d['newOrigZero'] = (np.isclose(d['newbalanceOrig'], 0)).astype(int)
    d['oldDestZero'] = (np.isclose(d['oldbalanceDest'], 0)).astype(int)
    d['newDestZero'] = (np.isclose(d['newbalanceDest'], 0)).astype(int)
    d['logAmount'] = np.log1p(d['amount'])
    d['isTransfer'] = (d['type'] == 'TRANSFER').astype(int)
    d['isCashOut'] = (d['type'] == 'CASH_OUT').astype(int)
    d['rule_fraud_transfer'] = ((d['isTransfer'] == 1) &
                                (np.isclose(d['oldbalanceDest'], 0)) &
                                (np.isclose(d['orig_delta'], 0))).astype(int)
    d['rule_fraud_cashout'] = ((d['isCashOut'] == 1) &
                               (np.isclose(d['newbalanceOrig'], 0)) &
                               (np.isclose(d['amount'], d['oldbalanceOrg']))).astype(int)
    d['rule_fraud'] = ((d['rule_fraud_transfer'] == 1) | (d['rule_fraud_cashout'] == 1)).astype(int)
    return d

def load_model(path):
    return joblib.load(path)

def predict_rule_plus_model(df_raw: pd.DataFrame, model_path: str):
    model = load_model(model_path)
    d = add_features(df_raw)
    y_rule = d['rule_fraud'].astype(int).values
    amb_mask = (y_rule == 0)
    y_model = np.zeros_like(y_rule)
    if amb_mask.any():
        # Match training feature columns
        cat_cols = ['type']
        num_cols = [c for c in d.columns if c not in set(['isFraud','isFlaggedFraud','rule_fraud','rule_fraud_transfer','rule_fraud_cashout']) | set(cat_cols)]
        feature_cols = cat_cols + num_cols
        X_amb = d.loc[amb_mask, feature_cols]
        y_model_part = model.predict(X_amb)
        y_model[amb_mask] = y_model_part
    y_pred = np.where(y_rule==1, 1, y_model)
    return y_pred
"""

with open(OUT_DIR / 'inference.py', 'w') as f:
    f.write(infer_code)

print("Saved:", OUT_DIR)
